# Hybrid Search Langchain

In [114]:
from dotenv import load_dotenv
import os
load_dotenv()
from langchain_community.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain_huggingface import HuggingFaceEndpointEmbeddings
from langchain_pinecone import PineconeVectorStore, PineconeSparseVectorStore
from pinecone import Pinecone, ServerlessSpec
from pinecone_text.sparse import BM25Encoder
from pinecone import Pinecone
from langchain_pinecone import PineconeVectorStore, PineconeSparseVectorStore
from langchain_pinecone.embeddings import PineconeSparseEmbeddings
from langchain_huggingface import HuggingFaceEmbeddings
os.environ["PINECONE_API_KEY"] = os.getenv("PINECONE_API_KEY")


In [115]:
dense_index_name = "hybrid-dense"
if dense_index_name not in pc.list_indexes().names():
    pc.create_index(name="hybrid-dense", dimension=384, metric="dotproduct", spec=ServerlessSpec(cloud="aws", region="us-east-1") )

# ========== Create SPARSE Index ==========
sparse_index_name = "hybrid-sparse"
if sparse_index_name not in pc.list_indexes().names():
    pc.create_index( name=sparse_index_name, metric="dotproduct",vector_type="sparse", spec=ServerlessSpec( cloud="aws", region="us-east-1" ))

# Get both index objects
dense_index = pc.Index(dense_index_name)
sparse_index = pc.Index(sparse_index_name)

os.environ["HF_API_KEY"] = os.getenv("HF_API_KEY")
embeddings = HuggingFaceEmbeddings( model_name="sentence-transformers/all-MiniLM-L6-v2" )
BM25_Encoder =  BM25Encoder().default()
BM25_Encoder

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3057.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [116]:
sentence =["""Straive is a market leading Content and Data Technology company providing data services, subject 
matter expertise, & technology solutions to multiple domains. 
Data Analytics & Al Solutions, Data Al Powered Operations and Education & Learning form the core 
pillars of the company’s long-term vision. The company is a specialized solutions provider to business 
information providers in finance, insurance, legal, real estate, life sciences and logistics. Straive continues to 
be the leading content services provider to research and education publishers.""",
"""Data Analytics & Al Services: Our Data Solutions business has become critical to our client's success. 
We use technology and Al with human experts-in loop to create data assets that our clients use to power 
their data products and their end customers' workflows. As our clients expect us to become their future-fit 
Analytics and Al partner, they look to us for help in building data analytics and Al enterprise capabilities 
for them. """
                
                ]


In [117]:


# ========== Initialize Embeddings ==========
sparse_embeddings = PineconeSparseEmbeddings( model="pinecone-sparse-english-v0" )

# ========== Create Dense Vector Store ==========
dense_store = PineconeVectorStore(index=dense_index, embedding=embeddings, namespace='dense')

# ========== Create Sparse Vector Store ==========
sparse_store = PineconeSparseVectorStore( index=sparse_index, embedding=sparse_embeddings, namespace="sparse" )

# Add to both stores
dense_store.add_texts(sentence)
sparse_store.add_texts(sentence)

# ========== Hybrid Search ==========
def hybrid_search(query, k=5):
    """Search using both dense and sparse, combine results"""
    # Dense search (semantic)
    dense_results = dense_store.similarity_search(query, k=k)
    print("Dense Results (Semantic):")
    for doc in dense_results:
        print(f"  - {doc.page_content}")
    
    # Sparse search (keyword-based)
    sparse_results = sparse_store.similarity_search(query, k=k)
    print("\nSparse Results (Keyword-based):")
    for doc in sparse_results:
        print(f"  - {doc.page_content}")
    
    # Combine and deduplicate
    combined = list(set([doc.page_content for doc in dense_results + sparse_results]))
    print(f"\nCombined Results ({len(combined)} unique):")
    for doc in combined:
        print(f"  - {doc}")
    return combined

# ========== Use It ==========
hybrid_search("Explain about Straive", k=1)

Dense Results (Semantic):
  - Straive is a market leading Content and Data Technology company providing data services, subject 
matter expertise, & technology solutions to multiple domains. 
Data Analytics & Al Solutions, Data Al Powered Operations and Education & Learning form the core 
pillars of the company’s long-term vision. The company is a specialized solutions provider to business 
information providers in finance, insurance, legal, real estate, life sciences and logistics. Straive continues to 
be the leading content services provider to research and education publishers.

Sparse Results (Keyword-based):
  - Straive is a market leading Content and Data Technology company providing data services, subject 
matter expertise, & technology solutions to multiple domains. 
Data Analytics & Al Solutions, Data Al Powered Operations and Education & Learning form the core 
pillars of the company’s long-term vision. The company is a specialized solutions provider to business 
information 

['Straive is a market leading Content and Data Technology company providing data services, subject \nmatter expertise, & technology solutions to multiple domains. \nData Analytics & Al Solutions, Data Al Powered Operations and Education & Learning form the core \npillars of the company’s long-term vision. The company is a specialized solutions provider to business \ninformation providers in finance, insurance, legal, real estate, life sciences and logistics. Straive continues to \nbe the leading content services provider to research and education publishers.']